# VoiceGuard — train the AASIST detector (Colab / any GPU)

Runs the full pipeline: pull data from HF → generate content-matched MMS-TTS fakes → build
manifests → cache SSL features on GPU → train → evaluate. Output: `aasist_indicw2v.pt`,
which you download and drop into `backend/models/` locally.

**Runtime → Change runtime type → GPU (T4 is fine).** Free-tier session is enough for the
frozen-frontend baseline; use `STAGE2=True` on a longer/Pro session for the unfrozen model.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → GPU'
print(torch.cuda.get_device_name(0))
!pip -q install 'transformers>=4.44' 'datasets>=2.20' huggingface_hub soundfile librosa pyyaml

In [ ]:
# --- get the repo ---
REPO_URL = ''  # <-- your GitHub clone URL, e.g. https://github.com/<you>/voiceguard.git
import os
if REPO_URL:
    !git clone --depth 1 {REPO_URL} voiceguard
else:
    print('No REPO_URL set. Upload a repo zip via the Files pane and: !unzip -q voiceguard.zip')
%cd voiceguard
!ls

In [ ]:
# --- knobs ---
FRONTEND   = 'facebook/wav2vec2-xls-r-300m'  # or facebook/wav2vec2-base / ai4bharat/indicwav2vec-hindi
LAYER      = -1
FAKE_N     = 1200      # MMS-TTS clips per language
EPOCHS     = 20
STAGE2     = False     # unfreeze the frontend after (needs a longer session)
LIMIT      = None      # set e.g. 400 for a quick smoke run

import yaml, pathlib
cfg_path = pathlib.Path('training/config_train.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['device'] = 'cuda'
cfg['frontend']['model_id'] = FRONTEND
cfg['frontend']['layer'] = LAYER
cfg['frontend']['stage2_unfreeze_epoch'] = (EPOCHS // 2) if STAGE2 else None
cfg['epochs'] = EPOCHS
cfg['num_workers'] = 2
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
# --- run everything ---
args = f"--config training/config_train.yaml --fake-n {FAKE_N} --epochs {EPOCHS}"
if LIMIT: args += f" --limit {LIMIT}"
!python -m training.pipeline_run {args}

In [ ]:
# --- download the checkpoint ---
from google.colab import files
files.download('backend/models/aasist_indicw2v.pt')
# then locally:  move it to backend/models/aasist_indicw2v.pt  (the live pipeline auto-loads it)